In [1]:
%run ../../Latex_macros.ipynb

<IPython.core.display.Latex object>

# RLLM-5: PPO in depth

## Where this session fits

> **The limitation we start from: Intro session C1 showed PPO in outline. On a language model it takes four networks, a schedule and a critic that must keep up.**

- **The question:** what does PPO need in practice, where does it go wrong, and why does the clip keep it in check?
- **Position: session 5 of the RL for LLMs series (RLLM-1 to RLLM-13)**
    - RLLM-4 used GRPO; this session opens the PPO machinery behind RLHF (RLLM-2)
    - the next session, RLLM-6, is GRPO and its refinements
- **Series goal:** turn a pretrained language model into a useful, aligned assistant with reinforcement learning, and know the methods well enough to choose among them

**You are here**

<img src="images/series_map_5.png" width=90%>

**By the end you can**

1. name the models in a PPO run on a language model, and say which are updated, and when
2. write the clipped objective, and say what it does and why a small move per batch is wanted
3. compute per-token rewards, TD errors and GAE advantages for a short completion, and say what the critic learns from
4. say what the critic costs on a language model, and read the signals of a PPO run
5. compare PPO and GRPO on advantage, networks and credit

**Intro sessions we use, and do not repeat**

- B3 *Policy gradient III: a learned critic* (TD error, GAE)
- B4 *Policy gradient IV: how far may one update move?* (ratio, clip)
- C1 *Reinforcement learning on a language model* (per-token reward, KL)
- C2 *GRPO: a baseline without a critic*


## Words from earlier sessions

**Why this slide?** The Intro series did not define these five words, and the rest of the session uses them.

1. **SFT** (supervised fine-tuning): more training of a pretrained model on prompts with good completions (RLLM-1)
2. **reward model**: a network that reads a prompt and a completion and returns one number, the reward (RLLM-2)
3. **RLHF** (reinforcement learning from human feedback): RL whose reward comes from a reward model trained on human comparisons
4. **proxy**: a measure we optimize because the true goal, quality, is hard to measure
5. **prefix**: the prompt plus the tokens written so far, playing the role of the state


## The models in a PPO run

**Why this section?** The first source of confusion is how many networks there are, and which ones learn.

> **Four networks matter: the policy and the critic are updated; the reference model and the reward model are frozen.**

<img src="images/rllm5_models.png" width=75%>

| Model | Input → output | Updated? | Role |
|:--|:--|:--|:--|
| **policy** $\pi_\theta$ | prefix → next-token distribution | yes | writes the completions |
| **critic** $\statevalfun_\phi$ | prefix → one number | yes | estimates the reward still to come: the baseline |
| **reference** $\pi_{\text{ref}}$ | prefix → next-token distribution | no | the SFT model: anchor of the KL reward |
| **reward model** | prompt and completion → one number | no | the reward at the last token (a program, if verifiable) |

**One more thing to place: the old policy**

- the **old policy** $\pi_{\text{old}}$ (Intro session B4) is the policy that generated the current batch
    - a snapshot, not a fifth network; its log-probabilities are stored during sampling
- the policy and the reference start as copies of the SFT model
- the critic is often the same network with a new output layer, or starts from the reward model


## One iteration, in order

**Why this section?** The schedule explains why some models are fresh and some are stale.

> **PPO alternates a costly generation phase with several cheap update epochs on the same batch.**

<img src="images/rllm5_loop.png" width=80%>

- **rollout:** generate completions from the current policy
    - usually the main cost, so the batch is reused
- **rewards:** one reward-model reward per completion, and a KL term for every token (C1)
- **advantages:** the critic values every prefix; GAE (B3) turns values and rewards into advantages
- **$K$ epochs:** $K$ passes over the same batch
    - each pass is split into **mini-batches**, small slices
    - each slice updates the policy (clipped) and the critic
- **refresh:** the updated policy samples the next batch

**Why the batch goes stale**

> **After the first mini-batch the policy is no longer the one that wrote the batch, so the batch describes a policy we have left.**

- supervised learning has fixed labels; here the data is the model's own earlier behavior
- two things go stale
    - the **samples**: they come from the old policy, and the ratio corrects for this only while the policies are close
    - the **critic**: it was fit to the policy of the last iteration
- the fix: keep the new policy near the old one
- the next two slides show how, and why


## The clip: a brake on each token

**Why this slide?** The batch is reused for $K$ epochs, so something must stop the policy from running away from it.

> **PPO stops pushing a token once its probability has moved by more than $\epsilon$ (typically $0.2$) from the old policy's, in the direction the advantage asks for.**

- the ratio of the sampled token is $r_{\tt}(\theta) = \pi_\theta(\text{token}_{\tt} \mid \text{prefix}_{\tt}) \,/\, \pi_{\text{old}}(\text{token}_{\tt} \mid \text{prefix}_{\tt})$

$$L^{\text{clip}}(\theta) = \E_{\tt} \left[ \operatorname{min} \left( r_{\tt}(\theta)\, \advseq_{\tt},\; \operatorname{clip}\left(r_{\tt}(\theta),\, 1-\epsilon,\, 1+\epsilon\right)\, \advseq_{\tt} \right) \right]$$

- the policy parameters $\theta$ move to increase $L^{\text{clip}}$

<img src="images/rllm5_clip.png" width=88%>

- left: a good token ($A > 0$) is pushed up until $r$ reaches $1 + \epsilon$
    - then the objective is flat and the gradient is zero
- right: a bad token ($A < 0$) is pushed down until $r$ reaches $1 - \epsilon$
- the `min` keeps the pessimistic of the two curves, so a token that moved the wrong way is still corrected
- schematic, for one token; the ratio can still overshoot the range within a mini-batch


## Why keep the move small: the surrogate drifts

**Why this slide?** The clip is a brake; this is the reason for braking.

> **The objective is a stand-in (the surrogate) that is exact at the old policy and drifts from the true improvement as the policy moves.**

- **KL** (Kullback-Leibler divergence, Intro session C1): how far apart two distributions are
    - zero when they are equal
- reason: the batch shows the states the *old* policy visited
    - the new policy would visit others (for a language model, other prefixes)
- **measured** in Intro session B4: 40 random small MDPs, exact improvement known, the policy moved along the gradient


<img src="images/rllm5_drift.png" width=58%>

- the error grows with the KL
- it varies by problem: at KL $0.1$, the middle 80% of problems had an error of 2% to 22% of the true improvement
- synthetic problems, as measured in B4; a language model has no such exact check

**From a measured drift to a clip**

> **The aim is to keep the KL between old and new policy small. TRPO enforces that directly; PPO's clip is a cheaper way to the same end.**

- **KL** (Intro session C1) measures how far one distribution is from another
- **TRPO** (Schulman et al. 2015) bounds that KL directly, with second-order machinery
- PPO limits only the ratio of the *sampled* token, with plain first-order steps
- **our reading:** the clip stands in for the KL bound but does not enforce it, so there is no guarantee
- the **clip fraction** (below) shows how often the brake is on
- the exact identity and TRPO's bound are in the asides at the end


## The per-token reward

**Why this section?** In RLHF-style PPO every token gets a reward, though the reward model speaks only at the end.

> **The reward at token $\tt$ is the KL term $-\beta \log(\pi_\theta / \pi_{\text{ref}})$, plus the reward model's reward if $\tt$ is the last token.**

- this is Intro session C1's per-token reward
- the critic and GAE then treat it like any reward sequence
- the KL term is **sampled**: the log-ratio of the token that was drawn, not the full KL over the vocabulary
    - summed over a completion it estimates $\beta \, \KL(\pi_\theta \,\|\, \pi_{\text{ref}})$
    - a single token's term can be positive or negative

**What $\beta$ does, and a symbol clash to watch**

- $\beta$ is the **KL coefficient**: it sets the leash on the policy relative to the *reference*
    - large: the policy stays near the reference, and the reward model has less influence
    - small: the reward model dominates, and the proxy is easier to exploit
- three symbols are reused; a flag tells them apart
    - **$\beta$**: the KL coefficient of the reward (TRPO's penalty is $C$, in the aside)
    - **$\delta_{\tt}$**: the TD error; TRPO's KL bound is $\delta_{\text{KL}}$
    - **$r_{\tt}(\theta)$**: the PPO ratio; the reward is $\rew_{\tt}$


## Worked example: credit for each token

**Why this section?** We follow one short completion through the machinery, with a critic and the KL reward.

> **With a critic, each token gets its own advantage, computed from the critic's surprise at each step.**

- a four-token completion, $\gamma = 1$, $\beta = 0.1$; the reward model's reward is $0.2$
- before token 1 the critic expects $0.30$ in total
    - the completion earns $-0.05$: KL rewards $-0.25$, reward model $+0.2$
    - so it is well below its expectation, by $0.35$
- the value after the last token is $0$: the completion has ended
- the **TD error** is $\delta_{\tt} = \rew_{\tt} + \gamma\,\statevalfun_\phi(\text{next prefix}) - \statevalfun_\phi(\text{prefix})$ (B3)

| | token 1 | token 2 | token 3 | token 4 |
|:--|:--|:--|:--|:--|
| log-ratio $\log \pi_\theta / \pi_{\text{ref}}$ | 1.0 | 0.0 | 0.5 | 1.0 |
| KL reward $-\beta \times$ log-ratio | -0.10 | 0.00 | -0.05 | -0.10 |
| reward model's reward (last token only) | 0 | 0 | 0 | +0.20 |
| reward $\rew_\tt$ (the two rows above, added) | -0.10 | 0.00 | -0.05 | +0.10 |
| critic value $\statevalfun_\phi$ of the prefix before the token | 0.30 | 0.50 | 0.45 | 0.45 |
| TD error $\delta_\tt$ | +0.10 | -0.05 | -0.05 | -0.35 |
| advantage, $\lambda = 0.95$ | -0.29 | -0.41 | -0.38 | -0.35 |

<img src="images/rllm5_gae.png" width=95%>

- left: what the critic expected from each prefix, against the reward that actually came after it
- $\lambda = 1$ gives *reward still to come minus the critic's value*: every token is told about the shortfall
    - tokens 2 and 3 are blamed most: the critic valued their prefixes at $0.50$ and $0.45$, and only $0.05$ followed
- $\lambda = 0$ gives the TD error: token 1 is **praised** ($+0.10$), because the critic liked the next prefix better
- $\lambda = 0.95$ lies between: token 1 gets $-0.29$, blamed through the later errors it leads to, though its own TD error is positive
- real completions are hundreds of tokens, so the choice matters more there
- **no critic, no choice:** GRPO gives all four tokens the same number (Intro session C2)


## What the critic learns from

**Why this slide?** The advantages above are only as good as the critic, so we need to know what it is trained to match.

> **The critic is trained by regression to a target built from this batch: the token's advantage plus the critic's own earlier value.**

- target $\hat R_{\tt} = \advseq_{\tt} + \statevalfun_\phi(\text{prefix}_{\tt})$, with the values taken at rollout time and held fixed
- **value loss:** the mean squared gap between the critic's current value and $\hat R_{\tt}$
- the TD-target idea of Intro session B3, run on every prefix of the batch, for $K$ epochs

| | token 1 | token 2 | token 3 | token 4 |
|:--|:--|:--|:--|:--|
| advantage $\advseq_\tt$, $\lambda = 0.95$ | -0.29 | -0.41 | -0.38 | -0.35 |
| critic's value before the update | 0.30 | 0.50 | 0.45 | 0.45 |
| **target** $\hat R_\tt$ = advantage + value | +0.01 | +0.09 | +0.07 | +0.10 |

- with $\lambda = 1$ the target is exactly the reward still to come
- with $\lambda < 1$ it blends in the critic's own next-prefix value


## What the critic costs on a language model

**Why this section?** The critic's credit assignment is not free: this cost motivates GRPO.

> **The critic is a second network, trained from a start where its values are meaningless, that must keep up with a policy that keeps moving.**

- **memory and compute:** its own parameters, gradients and optimizer state
    - a critic of the policy's size roughly doubles what is trained
    - our estimate; a smaller critic costs less
- a **value head** is an extra output layer that gives the critic's number
    - on a shared backbone it saves memory
    - one network, two outputs; but the two gradients compete for the same parameters
- **the cold start:** the critic's head begins random, so early advantages are noisy
    - some implementations train only the critic first, or start it from the reward model

**A critic that falls behind**

- the policy changes, and the critic's targets change with it
- a lagging critic gives biased or noisy advantages, and the policy then follows noise
- two reasons it is hard on long completions
    - one reward at the end must be spread over hundreds of tokens
    - the value of an early prefix averages many possible continuations, and few samples show it
- long reasoning chains are a known failure case for the critic
    - two 2025 papers on it: VC-PPO ([2503.01491](https://arxiv.org/abs/2503.01491)) and VAPO ([2504.05118](https://arxiv.org/abs/2504.05118)); not covered here
- RLLM-6 shows the alternative: no critic at all


## Reading a PPO run: what to watch

**Why this section?** A run fails quietly; the signals tell you which part is failing.

> **A rising reward with a growing KL can mean the proxy is being exploited, not that the assistant is getting better.**

<img src="images/rllm5_signals.png" width=92%>

- **synthetic illustration** (hand-made curves, not a real run): the reward model's reward rises while true quality peaks and then falls
- a larger $\beta$ slows this, and probably the real improvement too (our inference)
- reward-model over-optimization (RLLM-2) has the same shape

**The signals, and what a warning looks like**

| Signal | What it measures | A warning looks like (rules of thumb, our reading) |
|:--|:--|:--|
| mean reward | what the policy optimizes | rises while real quality does not |
| KL to the reference | how far the policy has moved | grows without bound, or stays at $0$ |
| value loss | the critic's squared gap to its target | high, not falling: advantages are noise |
| clip fraction | share of tokens that hit the clip | very high: steps too large; very low: too timid |
| entropy | spread of the next-token distribution (0 = one token) | collapses early: exploration is gone |
| completion length | what is being produced | grows: the reward may favor length |


## PPO and GRPO, side by side

**Why this section?** This is the trade at the heart of Intro session C2.

> **PPO buys per-token credit with a critic. GRPO gives up per-token credit to remove it.**

| | PPO | GRPO |
|:--|:--|:--|
| **baseline** | learned: the critic's value of each prefix | computed: the mean reward of the group |
| **advantage** | per token (GAE) | one per completion, shared by its tokens |
| **networks in training** | policy and critic | policy only |
| **KL term** | a reward at every token | a loss term (RLLM-6) |
| **needs** | a critic that tracks the policy | many completions per prompt, with varied rewards |
| **natural when** | per-token credit matters: dense or shaped rewards | a whole completion is judged: verifiable rewards |

<img src="images/rllm5_credit.png" width=70%>

- the worked example again: PPO tells the four tokens apart; GRPO cannot
- the GRPO bar is only illustrative
    - real GRPO computes its number from the group's rewards (RLLM-4), not from these four tokens
- GRPO spends compute on many completions per prompt instead of on a critic


# What we established, and what it sets up

- **Established**
    - four networks: policy and critic learn; reference and reward model are frozen
    - one iteration: roll out, add the KL reward, compute advantages with the critic and GAE, update for $K$ epochs, refresh
    - the clip brakes a token once its ratio leaves $[1-\epsilon, 1+\epsilon]$: our reading is a stand-in for a KL bound, with no guarantee
    - the critic gives each token an advantage and costs a second network that must keep up
    - the run signals: reward, KL, value loss, clip fraction, entropy, length
- **Sets up**
    - **the limitation:** a critic is costly and fragile on long completions, and GRPO's shared advantage is coarse
    - next session, RLLM-6, *GRPO and its refinements*, takes GRPO's known weak spots one by one


## Aside: the exact identity, and TRPO's bound

> **The surrogate gets the new policy's actions right but the states wrong. TRPO bounds the error by the KL between the policies.**

- the exact identity (the performance difference lemma, Kakade and Langford 2002)
    - the true improvement is the average of the *old* advantages, over the *new* policy's actions **and the states the new policy visits**
- the surrogate keeps the new policy's actions (through $r$) but uses the states the *old* policy visited
    - so it is exact at $\theta = \theta_{\text{old}}$; the whole error comes from the change in visited states
- TRPO (Schulman et al. 2015) bounds that error by a constant times the KL between old and new policy
    - the theorem uses the largest KL over states; the practical algorithm uses the average
    - the constant grows quickly with the horizon, so the bound is a reason to keep the KL small, not a usable size of the error


## Aside: TRPO as a constraint, and as a penalty

- **TRPO:** maximize the surrogate subject to a bound on the KL between old and new policy
    - $\displaystyle \max{\theta} \;\E_{\tt} \left[ r_{\tt}(\theta)\, \advseq_{\tt} \right] \quad \text{subject to} \quad \E_{\state} \left[ \KL \left( \pi_{\theta_{\text{old}}}(\cdot \mid \state) \,\|\, \pi_\theta(\cdot \mid \state) \right) \right] \leq \delta_{\text{KL}}$
    - the **trust region** is the set of policies where we trust the local surrogate
    - **flag:** $\delta_{\text{KL}}$ is TRPO's bound, not the TD error $\delta_{\tt}$
- **the same trade-off as a penalty**, with coefficient $C$ (the TRPO paper's symbol)
    - $\displaystyle \max{\theta} \;\E_{\tt} \left[ r_{\tt}(\theta)\, \advseq_{\tt} \right] \;-\; C \, \E_{\state} \left[ \KL \left( \pi_{\theta_{\text{old}}}(\cdot \mid \state) \,\|\, \pi_\theta(\cdot \mid \state) \right) \right]$
    - **flag:** $C$ penalizes old versus new policy; the RLHF coefficient $\beta$ penalizes policy versus *reference*
- the theory asks for a $C$ so large that steps would be tiny, so the practical TRPO uses the constraint, which carries no such guarantee


## Aside: the KL direction in TRPO

- $\KL(p \,\|\, q) = \E_{x \sim p}\bigl[ \log p(x) - \log q(x) \bigr]$ is not symmetric in $p$ and $q$
- TRPO writes $\pi_{\theta_{\text{old}}}$ first: the expectation is over the *old* policy, the one that generated the data
- an action the new policy introduces but the old one never took has weight $0$ in this expectation
    - so it has no *direct* effect on the constraint, only an indirect one through the probability it takes from other actions
- the KL reward of RLHF has the other direction: $\KL(\pi_\theta \,\|\, \pi_{\text{ref}})$, averaged over the current policy
